<a href="https://colab.research.google.com/github/rgomesa2025/MVP_MACHINE_LEARNING_E_ANALYTICS/blob/main/notebook/Producao_Nacional_Petroleo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — Machine Learning & Analytics

**PUC-RIO** - Pós-Graduação em Ciência de Dados e Analytics

**Nome:** _Rosângela Gomes André_

**Matrícula:** _4052025002132_

**Data:** _05/07/2026_

**Dataset:** _[Produção de Petróleo e gás natural (ANP)](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos) / [Produção por Poços](https://cdp.anp.gov.br/ords/r/cdp_apex/consulta-dados-publicos-cdp/consulta-produ%C3%A7%C3%A3o-por-po%C3%A7o)_

**Tipo de problema:** _Séries Temporais_

# 1. Definição do Problema

## 1.1 Contexto do Problema
A indústria de petróleo e gás gera um volume massivo de dados sobre a extração de recursos. O acesso a essas informações de forma pública permite analisar o histórico de produção dos principais campos nacionais, mas olhar apenas para o passado não basta para o planejamento de longo prazo. O desafio está em entender o comportamento futuro da produção diante da oscilação natural dos reservatórios.

### Previsão que o Modelo Deve Apoiar
O modelo de Machine Learning deve apoiar a **previsão de comportamento para os próximos 5 anos** de cinco variáveis fundamentais e interdependentes do dataset público:
1. Óleo
2. Óleo Condensado
3. Gás Natural
4. Gás Natural Associado
5. Gás Natural Não Associado

A solução vai gerar projeções volumétricas futuras para cada uma dessas curvas simultaneamente.

### Relevância do Problema
Prever a produção de hidrocarbonetos com um horizonte de 5 anos é crucial porque as decisões nesse setor envolvem investimentos de altíssimo valor e longo tempo de maturação. Uma previsão imprecisa de gás (associado ou não associado), por exemplo, pode resultar em falta de capacidade de escoamento ou gasodutos ociosos. Antecipar essas curvas reduz incertezas financeiras, otimiza a logística de transporte e garante maior segurança operacional e estratégica para o mercado de energia.

## 1.2 Objetivo do MVP

O objetivo deste MVP é desenvolver e avaliar modelos de Machine Learning capazes de projetar as curvas de produção de longo prazo (5 anos) para óleo, óleo condensado, gás natural, gás associado e gás não associado a partir do histórico de dados públicos do setor. A solução visa comparar o desempenho de algoritmos preditivos contra uma abordagem de referência (baseline), analisando a viabilidade técnica das projeções e mapeando as limitações operacionais do modelo.

## 1.3 Tipo de problema

* **Tipo escolhido:** Séries temporais / forecasting (combinado com Regressão).

* **Justificativa:** O projeto consiste na previsão de valores numéricos contínuos (volumes de produção de óleo e gás) onde a ordem cronológica dos dados é fundamental para o aprendizado do modelo. Não se trata de uma regressão simples ou classificação, pois o comportamento futuro das curvas depende diretamente do histórico temporal anterior. Por se tratar de um problema temporal, o ordenamento dos dados será estritamente respeitado na separação de treino e teste, evitando técnicas tradicionais de embaralhamento (shuffle) que causariam vazamento de dados do futuro para o passado.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. A produção histórica recente e o comportamento de declínio natural dos reservatórios contêm padrões sazonais e de tendência suficientes para que algoritmos de Machine Learning projetem o comportamento futuro em um horizonte de 5 anos.
2. Existe uma correlação e interdependência significativa entre as 5 curvas (especialmente entre o óleo e o gás associado), permitindo que o modelo capture dinâmicas conjuntas de produção e entregue previsões mais consistentes do que projeções isoladas.
3. Modelos avançados de séries temporais ou de regressão baseados em árvores (como XGBoost ou LightGBM adaptados para contexto temporal) apresentarão um desempenho superior a uma abordagem estatística simples (baseline) para previsões de longo prazo.

**Critérios de sucesso:**
- **Métrica principal:** MAPE (Erro Percentual Absoluto Médio) e RMSE (Raiz do Erro Quadrático Médio). O MAPE será a métrica de negócio para avaliar o percentual médio de erro em cada uma das 5 curvas.
- **Resultado mínimo esperado:** Reduzir o MAPE em pelo menos 15% em comparação com o modelo baseline (que será definido pela repetição do último valor observado ou por uma média móvel simples).
- **Restrição prática:** Simplicidade e custo computacional. Como se trata de um MVP para previsão de longo prazo (estratégica, não em tempo real), o foco principal será a interpretabilidade das variáveis e a estabilidade do modelo ao longo dos 5 anos projetados, garantindo que o pipeline possa rodar sem a necessidade de infraestrutura de nuvem de altíssimo custo.

# 2. Ambiente, bibliotecas e reprodutibilidade

Esta seção reúne o ecossistema de software, bibliotecas e as configurações de semente (*seed*) necessárias para garantir que a execução deste notebook produza exatamente os mesmos resultados em qualquer ambiente compatível.

### Bibliotecas Utilizadas
O pipeline foi desenvolvido utilizando a linguagem Python, apoiando-se nos seguintes pacotes:
* **Manipulação e Análise de Dados:** `pandas` (para engenharia de recursos e estruturação das séries temporais) e `numpy` (para operações matemáticas de suporte).
* **Visualização de Dados:** `matplotlib` e `seaborn` (fundamentais para a análise exploratória das curvas de declínio e plotagem das projeções de 5 anos).
* **Modelagem Preditiva e Machine Learning:** `scikit-learn` (para divisão temporal dos dados, pré-processamento e cálculo de métricas como RMSE) e `xgboost` / `lightgbm` (algoritmos candidatos para a regressão multivariada adaptada ao contexto temporal).

### Reprodutibilidade (Seed Fixa)
Para garantir que as divisões de validação e a inicialização de determinados algoritmos não variem entre execuções, uma semente aleatória global foi definida.



In [3]:
#python
import os
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Validação e Divisão de Dados Temporais
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Modelos (Baseline e Candidatos Avançados)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Métricas de Avaliação para Regressão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import randint

warnings.filterwarnings("ignore")

# Configuração de semente fixa para reprodutibilidade
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Seed:", SEED)

Python: 3.12.13
Seed: 42


## 2.1 Dependências adicionais

Para a execução deste MVP, é necessária a instalação de pacotes complementares ao ecossistema padrão do Python. Embora as bibliotecas de manipulação de dados e regressão linear básica (como Pandas e Scikit-Learn) já sejam nativas em ambientes de ciência de dados, a modelagem preditiva das 5 curvas de hidrocarbonetos exige algoritmos avançados de Boosting.

Portanto, o projeto faz uso das seguintes dependências externas para garantir o treinamento dos modelos candidatos e a qualidade visual dos gráficos de projeção:



In [4]:
#python
# Executar a linha abaixo caso o ambiente não possua os pacotes instalados:
!pip install -q xgboost lightgbm seaborn

## 2.2 Funções auxiliares

Para garantir a organização do pipeline e evitar a repetição de código durante a fase de modelagem, foram desenvolvidas funções específicas para a avaliação do desempenho preditivo das curvas. Como o MVP lida com um problema de regressão temporal multivariada, as funções calculam os erros volumétricos com base nas quatro principais métricas de aderência do mercado de energia: MAE, RMSE, R² e o MAPE (que quantifica o percentual médio de erro diretamente no negócio).

O bloco abaixo centraliza essas funções para que possam ser chamadas de forma limpa na validação do modelo baseline e dos modelos candidatos:


In [6]:
#python
def calculate_mape(y_true, y_pred):
    """
    Calcula o Erro Percentual Absoluto Médio (MAPE).
    Remove valores nulos ou zerados do histórico para evitar divisões por zero.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_regression(y_true, y_pred):
    """
    Centraliza o cálculo das métricas de sucesso estabelecidas para as 5 curvas
    (Óleo, Condensado, Gás Natural, Gás Associado e Não Associado).
    """
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "MAPE (%)": calculate_mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


def show_results_table(results_dict):
    """
    Consolida as métricas dos experimentos em um DataFrame do Pandas,
    permitindo a comparação direta entre o modelo baseline e os algoritmos de Machine Learning.
    """
    return pd.DataFrame(results_dict).T

# 3. Seleção e carga dos dados

Esta seção detalha a origem dos dados públicos utilizados no MVP, as justificativas para a escolha da base, os aspectos regulatórios envolvidos e a implementação do pipeline de extração automatizada das informações a partir do repositório do projeto.

### 3.1 Fonte dos dados

* **Nome do dataset:** Dados de Produção de Petróleo e Gás Natural.
* **Link da fonte:** [Produção de Petróleo e gás natural (ANP)](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos) / [Produção por Poços](https://cdp.anp.gov.br/ords/r/cdp_apex/consulta-dados-publicos-cdp/consulta-produ%C3%A7%C3%A3o-por-po%C3%A7o).
* **Por que este dataset foi escolhido:** O conjunto de dados concentra o histórico oficial, consolidado e auditado da produção nacional de hidrocarbonetos. Ele fornece, com granularidade ideal, o comportamento volumétrico real e desagregado das cinco curvas que compõem o escopo preditivo deste MVP, servindo como base sólida para o treinamento de algoritmos de séries temporais de longo prazo.
* **Restrições e condições consideradas:** O recorte dos dados considera o histórico consolidado mensal para evitar ruídos de leituras diárias ou paradas operacionais momentâneas de curtíssimo prazo (como manutenções de poços de 24h ou 48h), focando estritamente na tendência macro de declínio e comportamento dos reservatórios para o horizonte de 5 anos. Para viabilizar o pipeline automatizado deste MVP, a série histórica foi organizada de forma cronológica e particionada em arquivos anuais dentro do repositório GitHub do projeto.
* **Ética, privacidade, confidencialidade e licença:** O dataset é classificado como **Dados Abertos Governamentais**, sob a licença *Creative Commons Attribution (CC BY)*. Por se tratar de dados de produção agregados por campos institucionais, não existem informações sensíveis de indivíduos, violações de privacidade ou quebras de confidencialidade comercial, respeitando integralmente os preceitos éticos e as diretrizes da LGPD (Lei Geral de Proteção de Dados).

### 3.2 Carga dos dados

O bloco abaixo automatiza a extração e a consolidação do histórico de dados de produção. Para respeitar a organização do repositório no GitHub, onde as tabelas públicas estão divididas em pastas anuais e particionadas em arquivos mensais utilizando a nomenclatura padrão `ANO_MES_producao_Mar.csv` (onde "Mar" identifica o ambiente de produção offshore), o script realiza uma varredura cronológica gerando os índices dos meses de forma dinâmica e unifica todas as informações em um único DataFrame estruturado.



In [38]:
# URL RAW correta do GitHub
base_url = "https://raw.githubusercontent.com/rgomesa2025/MVP_MACHINE_LEARNING_E_ANALYTICS/main/data"

# Anos disponíveis no projeto (2021 a 2025)
anos_producao = ["2021", "2022", "2023", "2024", "2025"]

# Meses de 01 a 12
meses_producao = [str(i).zfill(2) for i in range(1, 13)]

dfs_mensais = []
arquivos_lidos = 0
arquivos_com_erro = 0

print("=== Iniciando consolidação dos arquivos ===")

for ano in anos_producao:
    print(f"Processando ano {ano} ", end="", flush=True)

    for mes in meses_producao:
        nome_arquivo = f"{ano}_{mes}_producao_Mar.csv"
        url_completa = f"{base_url}/{ano}/{nome_arquivo}"

        try:
            # Estratégia de leitura adaptada para padrões da ANP (Vírgula vs Ponto e Vírgula / UTF-8 vs Latin-1)
            try:
                # 1ª Tentativa: Padrão internacional (Vírgula + UTF-8)
                df_mes = pd.read_csv(url_completa, encoding='utf-8')
            except Exception:
                try:
                    # 2ª Tentativa: Padrão nacional Excel (Ponto e vírgula + Latin-1 para os acentos)
                    df_mes = pd.read_csv(url_completa, sep=';', encoding='iso-8859-1')
                except Exception:
                    # 3ª Tentativa: Variação de exportação (Vírgula + Latin-1)
                    df_mes = pd.read_csv(url_completa, sep=',', encoding='iso-8859-1')

            # Criação das colunas de controle temporal para a modelagem de séries
            df_mes["ano_referencia"] = int(ano)
            df_mes["mes_referencia"] = int(mes)

            dfs_mensais.append(df_mes)
            arquivos_lidos += 1
            # Imprime um ponto para indicar progresso visual sem inundar a tela
            print(".", end="", flush=True)

        except Exception as e:
            arquivos_com_erro += 1
            print(f"\n   ✗ Erro ao ler {nome_arquivo} | Motivo: {e}")

    print(" ✓ Concluído!")

print("\n=== Resumo da carga ===")
print(f"Arquivos carregados com sucesso: {arquivos_lidos}")
print(f"Arquivos não localizados ou com erro: {arquivos_com_erro}")

# Consolidação final unificando todas as tabelas mensais
if len(dfs_mensais) > 0:
    df = pd.concat(dfs_mensais, ignore_index=True)
    print("\n=== Consolidação concluída ===")
    print(f"Linhas totais no DataFrame: {df.shape[0]}")
    print(f"Colunas totais no DataFrame: {df.shape[1]}")
    print("\nPrimeiras linhas do dataset consolidado:")

    # Renderiza as linhas do DataFrame no formato de tabela visual HTML
    display(df.head())
else:
    print("\n[ERRO CRÍTICO] Nenhum arquivo foi carregado. Verifique a integridade dos dados no GitHub.")

=== Iniciando consolidação dos arquivos ===
Processando ano 2021 ............ ✓ Concluído!
Processando ano 2022 ............ ✓ Concluído!
Processando ano 2023 ............ ✓ Concluído!
Processando ano 2024 ............ ✓ Concluído!
Processando ano 2025 ............ ✓ Concluído!

=== Resumo da carga ===
Arquivos carregados com sucesso: 60
Arquivos não localizados ou com erro: 0

=== Consolidação concluída ===
Linhas totais no DataFrame: 61974
Colunas totais no DataFrame: 50

Primeiras linhas do dataset consolidado:


,Estado,Bacia,Nome Poço ANP,Nome Poço Operador,Campo,Operador,Número do Contrato,Período,Óleo (bbl/dia),Condensado (bbl/dia),...,% em Volumes Undecanos,% em Volumes Oxigênio,% em Volumes Nitrogênio,% em Volumes Gás Carbônico,Densidade GLP Gás,Densidade GLP Líquido,PCS GP(kJ/m³),Data de atualização,ano_referencia,mes_referencia
0,Rio de Janeiro,Santos,1-BRSA-976-RJS,1RJS691,SÉPIA,Petrobras,48610012913201005,2021/01,0,0,...,0,0,",60417","24,67724","2,03995","532,80966","29851,5187",09/06/2026,2021,1
1,Rio de Janeiro,Santos,3-BRSA-1201-RJS,3RJS721,SÉPIA,Petrobras,48610012913201005,2021/01,0,0,...,0,0,",60417","24,67724","2,03995","532,80966","29851,5187",09/06/2026,2021,1
2,Rio de Janeiro,Santos,9-BRSA-1254-RJS,9RJS733,SÉPIA,Petrobras,48610012913201005,2021/01,0,0,...,0,0,",60417","24,67724","2,03995","532,80966","29851,5187",09/06/2026,2021,1
3,Rio de Janeiro,Santos,9-SEP-1-RJS,9SEP1RJS,SÉPIA,Petrobras,48610012913201005,2021/01,0,0,...,0,0,",60417","24,67724","2,03995","532,80966","29851,5187",09/06/2026,2021,1
4,São Paulo,Santos,3-BRSA-788-SPS,3SPS69,SAPINHOÁ,Petrobras,486100038842000,2021/01,"19956,4647",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09/06/2026,2021,1


### 3.3 Visão geral do dataset

Antes de iniciar as transformações e a modelagem preditiva, esta subseção estabelece um diagnóstico estrutural completo do conjunto de dados unificado. A análise compreende o mapeamento volumétrico, a integridade das observações e a identificação do papel de cada variável dentro do pipeline de Machine Learning.

#### Análise Estrutural e Integridade dos Dados

O bloco de código abaixo executa a varredura técnica do DataFrame consolidado para extrair as propriedades fundamentais da base:



In [44]:
#python
# Verificação das dimensões do DataFrame
linhas, colunas = df.shape

print("=== Volumetria do Dataset ===")
print(f"Total de Registros (Linhas): {linhas}")
print(f"Total de Atributos (Colunas): {colunas}")

=== Volumetria do Dataset ===
Total de Registros (Linhas): 61974
Total de Atributos (Colunas): 50


In [55]:
#python

print("Formato do dataset:", df.shape)
print("\n=== Tipagem dos Atributos ===")
# Converte a série de tipos em um DataFrame com a coluna 'Tipo de Dado'
df_tipos = df.dtypes.to_frame(name="Tipo de Dado")

# Define o nome do índice
df_tipos.index.name = "Nome da Coluna"

# Transforma o índice em uma coluna comum para garantir a exibição dos títulos na tabela
df_tipos_visual = df_tipos.reset_index()

# Renderiza no formato visual de tabela HTML com os títulos explícitos
display(df_tipos_visual)


print("\n=== Resumo por Tipo de Dado ===")
# Converte o resumo numérico em um DataFrame com a coluna 'Quantidade de Colunas'
df_resumo = df.dtypes.value_counts().to_frame(name="Quantidade de Colunas")

# Define o nome do índice do resumo
df_resumo.index.name = "Tipo de Dado"

# Transforma o índice em uma coluna comum para o resumo
df_resumo_visual = df_resumo.reset_index()

# Renderiza a tabela de resumo formatada
display(df_resumo_visual)

Formato do dataset: (61974, 50)

=== Tipagem dos Atributos ===


,Nome da Coluna,Tipo de Dado
0,Estado,object
1,Bacia,object
2,Nome Poço ANP,object
3,Nome Poço Operador,object
4,Campo,object
5,Operador,object
6,Número do Contrato,object
7,Período,object
8,Óleo (bbl/dia),object
9,Condensado (bbl/dia),object



=== Resumo por Tipo de Dado ===


,Tipo de Dado,Quantidade de Colunas
0,object,47
1,int64,3


In [54]:
#python
print("=== Análise de Valores Ausentes (Nulos) ===")
total_nulos = df.isnull().sum()

# Peso percentual calculado dividindo os nulos da coluna pelo total de linhas da base (e não pela coluna vizinha)
proporcao_nulos = (df.isnull().sum() / len(df)) * 100

# Consolida em um DataFrame com nomenclaturas claras para o negócio
df_nulos = pd.DataFrame({
    'Total Nulos': total_nulos,
    'Proporção de Nulos (%)': proporcao_nulos.round(2)
})

# Define o nome do índice (que guarda os nomes das colunas originais)
df_nulos.index.name = "Nome da Coluna"

# Transforma o índice em uma coluna comum para garantir a exibição do título na tabela
df_nulos_visual = df_nulos.reset_index()

# Filtra para exibir apenas as colunas que possuem alguma ausência (Total Nulos > 0)
colunas_com_nulos = df_nulos_visual[df_nulos_visual['Total Nulos'] > 0]

if not colunas_com_nulos.empty:
    # Renderiza no formato visual de tabela HTML com as colunas: Nome da Coluna | Total Nulos | Proporção de Nulos (%)
    display(colunas_com_nulos)
else:
    print("Sucesso: Nenhum valor ausente encontrado no dataset consolidado!")


=== Análise de Valores Ausentes (Nulos) ===


,Nome da Coluna,Total Nulos,Proporção de Nulos (%)
16,Instalação Destino,10491,16.93
17,Tipo Instalação,10491,16.93
19,Período da Carga,7974,12.87
20,Corrente,7974,12.87
21,Grau API,7974,12.87
24,Fração de Destilados|Méd|Corte,7974,12.87
28,% em Volumes Metano,7616,12.29
29,% em Volumes Etano,7616,12.29
30,% em Volumes Propano,7616,12.29
31,% em Volumes Iso-Butano,7616,12.29


In [43]:

print("\n=== 4. Detecção de Linhas Duplicadas ===")
duplicate_count = df.duplicated().sum()
print(f"Total de registros completamente duplicados: {duplicate_count}")


=== 4. Detecção de Linhas Duplicadas ===
Total de registros completamente duplicados: 0
